In [ ]:
%pip install -q pandas==3.0.3 seaborn==0.13.2 scikit-learn==1.9.0 pooch

In [ ]:
import os, sys
# Skip clone/chdir when running under GitHub Actions (already in the right directory).
if not os.environ.get('CI'):
    if not os.path.exists('Xed'):
        os.system('git clone --depth=1 https://github.com/demianw/Xed.git')
    if 'ames_datasets' not in os.getcwd():
        os.chdir('Xed/ames_datasets')
# Make `xed.datasets` importable from any subdirectory.
_repo_root = os.path.abspath('..')
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

# Exploring Real Estate Sales Prices

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

**Learning objectives**

By the end of this notebook you will be able to:
1. Identify data quality issues (missing values, skewed distributions) in a real-world tabular dataset.
2. Build scikit-learn preprocessing pipelines combining imputation, scaling and encoding.
3. Evaluate regression models (Linear, Ridge, Lasso, Random Forest) using learning curves and cross-validation.
4. Interpret feature importance via permutation importance.

## 1. Exploration

### Question 1

Load the Ames housing dataset using `pandas`. It is located in `datasets/ames_housing.csv`. Using the function `head()` and `info()`, which issues do you identify which need to be noted before to learn a machine learning model.

The dataset is described in https://jse.amstat.org/v19n3/decock/DataDocumentation.txt

In [ ]:
data = load_ames_housing()
data.head()

### Question 2

- Identify the target variable: `SalePrice`, what is its type? What are its distributional characteristics?
- Which variables contain the most missing values?

## Question 3
Split the data into features and target variables.
Then, the data into a model selection, sample and a model evaluation sample. Use `sklearn.model_selection.train_test_split`.
Use a 20% ratio.

In [ ]:
from sklearn.model_selection import train_test_split
target = data["SalePrice"]
features = data.drop(columns="SalePrice")

selection_features, evaluation_features, selection_target, evaluation_target = train_test_split(
    features, target, test_size=.2
)
selection_target.shape

## Question 4
Extract the columns with numerical data using `selection_features.select_dtypes("number")`. Examine their distributions, through histograms. What issues do you identify? Then use  `selection_features.select_dtypes("number")` and seaborn's `sns.countplot` to analyze the string variables. Identify data types and issues.

# Section 2: Implement a linear regressor using only the numerical variables

### Question 1
Use a Column Transformer to _just select_ the numerical variables. Build a linear regressor using `sklearn.linear.LinearRegressor

For this we will
* build the column transformer
* build the machine learning pipeline
* evaluate it through cross-validation (using `cross_vals_score`)

Does it work? Why?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

### Question 2
Fix the previous issue using: first drop problematic rows, then use the `SimpleImputer`


In [ ]:
from sklearn.impute import SimpleImputer

### Question 3
Now plot the evolution of mean and standard deviations for test sample sizes of 05%, 10%, 20%, 25%, 30%. What do you conclude?


### Question 4
Use `sklearn.model_selection.learning_curve` to study the learning curve https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html?highlight=learning_curve

In [ ]:
from sklearn.model_selection import learning_curve

### Question 5
Are there correlations between the features? Explore it through the correlation matrix, and the `sns.pairplot` plotting tool from seaborn (warning, if you plot all variables together it might be slow)

### Question 6
Can we use this correlation to improve the learning curve? This is regularization, let's try ridge regression https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html

Plot the learning curve and compare it with the plain linear regression

In [ ]:
from sklearn.linear_model import Ridge

### Question 6.1
How did you pick your regularization parameter? Use a grid search now. https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html


In [ ]:
from sklearn.model_selection import GridSearchCV

### Question 7
Do we need all features? Repeat the previous analysis from Question 6 but with the Lasso which enforces sparsity https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html

### Question 8

Now we will repeat the same analysis but with the categorical variables. For which we will use the `OneHotEncoder` and the `OrdinalEncoder`
* https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
* https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html

and combine them in the preprocessing pipeline in Section 2, Questions 1 and 2.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

## Question 9

Non linearity! Now use the RandomForestRegressor https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html to fit and predict the data. The two hyper-parameters that you will use are

* n_estimators : with a default of 100 which deals with the uncertainty in the data/algorithm relationship.
* max_depth : with a no limit as a default which deals with the granularity of the solution.

Use a Grid search cross validation to set the two parameters. Plot the learning curve.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

## Question 10

We will now use the data to obtain
Use permutation feature importance to assess which are the most important features in predicting house pricing https://scikit-learn.org/stable/modules/permutation_importance.html

Compare these importances across models.

In [ ]:
from sklearn.inspection import permutation_importance

## Question 11

Pick one of the estimators. Use cross_val_predict to evaluate the quality of the prediction in different cases.

https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_predict.html#sphx-glr-auto-examples-model-selection-plot-cv-predict-py

Cross-val predict will give you for each element in the target, a prediction. Produce a scatterplot between target and prediction, use the trained model and the predictive importance to find the most explanatory variables.

In [ ]:
from sklearn.model_selection import cross_val_predict

# Section 3: Interpreting the best model

Now that you have selected a model, answer the following questions using
the tools introduced in Section 2:

### Question 1
Use `cross_val_predict` with your best pipeline to plot predicted vs. true
`SalePrice`. What does the scatter reveal about systematic prediction errors?

### Question 2
Compute permutation importances for the best pipeline on the evaluation set.
Which five features matter most? Do they align with domain intuition about
house prices?

### Question 3
Identify the five houses with the largest absolute prediction error.
Inspect their raw feature values — can you hypothesise why the model
struggled with them?

---
## Summary

In this notebook you:
- Explored the Ames housing dataset and diagnosed missing values and skewed features.
- Built preprocessing pipelines combining `SimpleImputer`, `StandardScaler`,
  `OneHotEncoder`, and `OrdinalEncoder` inside a `ColumnTransformer`.
- Compared Linear Regression, Ridge, Lasso, and Random Forest using learning curves.
- Used `GridSearchCV` to tune regularisation strength.
- Quantified feature contributions via permutation importance.

**Next notebook:** *Dimensionality Reduction* — apply PCA and Kernel PCA to the
same dataset and ask how many components are needed to retain predictive power.